# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya Exploration with `mlcroissant`
This notebook guides you in loading, exploring, and analyzing the FAIR² Mental Health Survey Dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json`

In [ ]:
# Ensure `mlcroissant` is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset using mlcroissant
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Dataset Name: {metadata.name}")
print(f"Description: {metadata.description}")
print(f"Identifier: {metadata.identifier}")
print(f"License: {metadata.license}")
print(f"Published: {metadata.datePublished}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

We will enumerate all available record sets and their associated fields, referencing them by their `@id`.

In [ ]:
record_sets = list(dataset.record_sets())  # Returns a list of RecordSet objects

print("Available Record Sets:")
for rs in record_sets:
    print(f"  RecordSet: {rs.name} (@id: {rs.id})")
    # List fields in this record set
    fields = rs.fields
    print("    Fields:")
    for field in fields:
        print(f"      {field.name} (@id: {field.id}) [type: {field.data_type}]")
    print()
    # If columns info exists, print it
    if hasattr(rs, 'columns') and rs.columns:
        print("    Columns:")
        for col in rs.columns:
            print(f"      {getattr(col, 'name', 'Unnamed')} (@id: {col.id})")
        print()
# Store the @id of the main record set for later use
main_record_set_id = record_sets[0].id if record_sets else None  # e.g. 'cr:RecordSet'

## 3. Data Extraction
Load data from the main record set into a pandas DataFrame for further analysis.

For all analyses, fields and columns will be referenced by their `@id`.

In [ ]:
# For demonstration, extract from all available record sets
dataframes = {}
record_set_ids = [rs.id for rs in record_sets]

print("Extracting record sets:", record_set_ids)

for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df

# Print the column list for the main record set
print(f"Columns for the main record set ({main_record_set_id}):")
print(dataframes[main_record_set_id].columns.tolist())

# Show sample records
dataframes[main_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)
We will:
- Select numeric fields for analysis (e.g., GAD-7 scores, PHQ-9 scores, PSQ scores).
- Filter records based on specific criteria.
- Normalize and group data.

**All field references are via their `@id` as defined in the Croissant schema.**

In [ ]:
# Example: Analyze GAD-7 scores
# Replace these IDs with the actual IDs from the overview above
numeric_field_id = None
group_field_id = None

# Identify likely numeric fields (by name)
numeric_fields_candidates = [col for col in dataframes[main_record_set_id].columns if 'gad' in col.lower() or 'phq' in col.lower() or 'psq' in col.lower()]

# Select first candidate for demo
if numeric_fields_candidates:
    numeric_field_id = numeric_fields_candidates[0]  # E.g. 'GAD-7_score' or its @id
    print(f"Using numeric field: {numeric_field_id}")
else:
    print("No obvious numeric field found.")

# Identify a grouping field, e.g., 'gender', 'marital_status', etc.
group_fields_candidates = [col for col in dataframes[main_record_set_id].columns if 'gender' in col.lower() or 'marital' in col.lower() or 'village' in col.lower()]
if group_fields_candidates:
    group_field_id = group_fields_candidates[0]
    print(f"Using group field: {group_field_id}")
else:
    print("No obvious group field found.")

# Filter the records where the GAD-7/PHQ-9/PSQ score (>10), handle missing values
threshold = 10
df = dataframes[main_record_set_id]

if numeric_field_id and numeric_field_id in df.columns:
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold}:")
    print(filtered_df.head())

    # Normalize the numeric field
    filtered_df["{}_normalized".format(numeric_field_id)] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Group by group_field (if available) and compute the mean
    if group_field_id and group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean()
        print(f"Grouped mean {numeric_field_id} by {group_field_id}:")
        print(grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships between fields. For example, we can plot the distribution of GAD-7 scores, or group statistics by `gender` or another demographic.

In [ ]:
# Plot numeric field distribution if available
if numeric_field_id and numeric_field_id in df.columns:
    plt.figure(figsize=(7,4))
    df[numeric_field_id].dropna().hist(bins=20)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Frequency")
    plt.show()

# Boxplot grouped by group_field
if numeric_field_id and group_field_id:
    if group_field_id in df.columns:
        plt.figure(figsize=(8,5))
        df.boxplot(column=numeric_field_id, by=group_field_id)
        plt.title(f"{numeric_field_id} distribution by {group_field_id}")
        plt.suptitle("")
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.show()

## 6. Conclusion
In this notebook, we've explored the FAIR² Mental Health Survey Dataset using `mlcroissant`, referenced all entities by their `@id`, and demonstrated exploratory analysis techniques.
Key points:
- The dataset provides valuable demographic and mental health indicators for Kilifi County, Kenya.
- Numeric assessment scores (e.g., GAD-7, PHQ-9) can be filtered, normalized, and grouped by demographic fields.
- Visualizations highlight score distributions and demographic splits.

Further analyses can focus on more advanced statistical comparisons or predictive modeling using the rich schema provided.